In [4]:
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

In [5]:
(X_train, y_train), (X_test, y_test) = (
    tf.keras.datasets.cifar10.load_data()
)

y_train = y_train.squeeze()
y_test = y_test.squeeze()

class_names = [
    "airplance", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"
]

print("Training Images : ", X_train.shape)
print("Test Images : ", X_test.shape)

Training Images :  (50000, 32, 32, 3)
Test Images :  (10000, 32, 32, 3)


In [6]:
X_val = X_train[-5000:]
y_val = y_train[-5000:]

X_train = X_train[:-5000]
y_train = y_train[:-5000]

def prepare_image(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (96, 96))
    image = image / 127.5 - 1.0
    return image, label

batch_size = 64

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(10000)
    .map(prepare_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .map(prepare_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .map(prepare_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [7]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(96, 96, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(96, 96, 3))

x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [8]:
outputs = layers.Dense(10, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

print("\nPHASE 1 : Feature Extraction")
model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)


PHASE 1 : Feature Extraction


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/3


/Users/jibanjyotimohanty/Jiban Drive/Machine-Learning/venv/lib/python3.14/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


704/704 ━━━━━━━━━━━━━━━━━━━━ 38s 52ms/step - accuracy: 0.8085 - loss: 0.5634 - val_accuracy: 0.8574 - val_loss: 0.4068
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 37s 53ms/step - accuracy: 0.8625 - loss: 0.3974 - val_accuracy: 0.8640 - val_loss: 0.3965
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 39s 55ms/step - accuracy: 0.8763 - loss: 0.3592 - val_accuracy: 0.8686 - val_loss: 0.3804


In [9]:
print("\nPHASE 1 : Fine Tuning")
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False


PHASE 1 : Fine Tuning


In [10]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

history_fine = model.fit(train_ds, validation_data=val_ds, epochs=2)

Epoch 1/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 50s 69ms/step - accuracy: 0.8950 - loss: 0.3034 - val_accuracy: 0.8764 - val_loss: 0.3539
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 49s 70ms/step - accuracy: 0.9127 - loss: 0.2561 - val_accuracy: 0.8786 - val_loss: 0.3557


In [11]:
test_loss, test_accuracy = model.evaluate(
    test_ds, verbose=1
)

print(f"Test Loss : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")

157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 54ms/step - accuracy: 0.8744 - loss: 0.3851
Test Loss : 0.3851
Test Accuracy : 0.8744


In [12]:
sample_image = X_test[0]

prepared_image, _ = prepare_image(
    sample_image, y_test[0]
)

probabilities = model.predict(
    tf.expand_dims(prepared_image, axis=0), verbose=0
)[0]

predicted_class = int(np.argmax(probabilities))

print("Actual Class : ", class_names[int(y_test[0])])
print("Predicted Class : ", class_names[predicted_class])
print("Probability For Each Class : ", np.round(probabilities, 3))

Actual Class :  cat
Predicted Class :  cat
Probability For Each Class :  [0.    0.    0.001 0.955 0.001 0.03  0.014 0.    0.    0.   ]
